In [182]:
### LOAD LIBRARIES ###

import pandas as pd
import numpy as np
import tensorflow 
from tensorflow import keras
from keras import layers
import os

In [183]:
### IMPORT DATA ###

wd = os.getcwd()

train = pd.read_csv(f"{wd}/airbnb/airbnb_train.csv")
X_train = train.drop("price", axis=1)
y_train = train["price"]

valid = pd.read_csv(f"{wd}/airbnb/airbnb_valid.csv")
X_valid = valid.drop("price", axis=1)
y_valid = valid["price"]

X_test = pd.read_csv(f"{wd}/airbnb/airbnb_test.csv")

# We dropped irrelevant variables based on the assumption that they would not be helpful for price prediction.
# ...

vars_to_drop = ['name', 'host_name', 'neighbourhood_group', 'latitude', 'longitude', 'availability_365']

X_train = X_train.drop(vars_to_drop, axis=1)
X_valid = X_valid.drop(vars_to_drop, axis=1)
X_test = X_test.drop(vars_to_drop, axis=1)

In [184]:
# import seaborn as sns

# sns.pairplot(train, diag_kind='kde')

# GABI: I think latitude and longitude could be merged, as well as the number of reviews with reviews per month.

In [185]:
### COMPUTE NEW VARIABLES ###

# ...

In [186]:
### PREPROCESS DATA ###

# Convert last_review to days before 2020-01-01
reference_date = pd.to_datetime('2020-01-01')

# Handle missing values in last_review before conversion
X_train['last_review'] = pd.to_datetime(X_train['last_review'], errors='coerce')
X_valid['last_review'] = pd.to_datetime(X_valid['last_review'], errors='coerce')
X_test['last_review'] = pd.to_datetime(X_test['last_review'], errors='coerce')

# Convert to days before reference date
X_train['last_review'] = (reference_date - X_train['last_review']).dt.days
X_valid['last_review'] = (reference_date - X_valid['last_review']).dt.days
X_test['last_review'] = (reference_date - X_test['last_review']).dt.days

vars_to_cat = ['host_id', 'neighbourhood', 'room_type']                 # List of categorical variables to hot encode
vars_to_norm = [i for i in X_train.columns if i not in vars_to_cat]     # List of numerical variables to normalize

# Standardize numerical variables

X_train_mean = np.mean(X_train[vars_to_norm])
X_train_std = np.std(X_train[vars_to_norm])

X_train[vars_to_norm] = (X_train[vars_to_norm] - X_train_mean) / X_train_std
X_valid[vars_to_norm] = (X_valid[vars_to_norm] - X_train_mean) / X_train_std
X_test[vars_to_norm] = (X_test[vars_to_norm] - X_train_mean) / X_train_std

# Hot encode categorical variables
for var in vars_to_cat:
    cats = X_train[var].astype("category").cat.categories
    X_train[var] = pd.Categorical(X_train[var], categories=cats).codes
    X_valid[var] = pd.Categorical(X_valid[var], categories=cats).codes
    X_test[var]  = pd.Categorical(X_test[var],  categories=cats).codes

# Missing values of review_per_month and last_review are tackled by substituting them with the variable median in order to keep the row count.
rpm_mean = X_train['reviews_per_month'].mean()
X_train['reviews_per_month'] = X_train['reviews_per_month'].fillna(rpm_mean)
X_valid['reviews_per_month'] = X_valid['reviews_per_month'].fillna(rpm_mean)
X_test['reviews_per_month']  = X_test['reviews_per_month'].fillna(rpm_mean)
lr_mean = X_train['last_review'].mean()
X_train['last_review'] = X_train['last_review'].fillna(lr_mean)
X_valid['last_review'] = X_valid['last_review'].fillna(lr_mean)
X_test['last_review'] = X_test['last_review'].fillna(lr_mean)

C:\Users\w20ce\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\numpy\_core\fromnumeric.py:4062: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)


In [ ]:
# Run basic model

vars = len(X_train.columns)

model = keras.Sequential(
    [
        layers.Dense(vars, activation="linear"),
        layers.Dense(16, activation="linear"),
        layers.Dense(1, activation="linear")
    ]
)

model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["root_mean_squared_error"],
)

history = model.fit(
    X_train,
    y_train,
    epochs=200,
    batch_size=64
)

results = model.evaluate(X_valid, y_valid)
print(f"RMSE: {results[1]:.4f}")

# RMSE: 87.57

Epoch 1/200
191/191 ━━━━━━━━━━━━━━━━━━━━ 1s 717us/step - loss: 38250.9375 - root_mean_squared_error: 195.5785
Epoch 2/200
191/191 ━━━━━━━━━━━━━━━━━━━━ 0s 648us/step - loss: 10586.9736 - root_mean_squared_error: 102.8930
Epoch 3/200
191/191 ━━━━━━━━━━━━━━━━━━━━ 0s 693us/step - loss: 10445.3301 - root_mean_squared_error: 102.2024
Epoch 4/200
191/191 ━━━━━━━━━━━━━━━━━━━━ 0s 702us/step - loss: 10287.0723 - root_mean_squared_error: 101.4252
Epoch 5/200
191/191 ━━━━━━━━━━━━━━━━━━━━ 0s 659us/step - loss: 10297.1934 - root_mean_squared_error: 101.4751
Epoch 6/200
191/191 ━━━━━━━━━━━━━━━━━━━━ 0s 656us/step - loss: 10251.1865 - root_mean_squared_error: 101.2481
Epoch 7/200
191/191 ━━━━━━━━━━━━━━━━━━━━ 0s 635us/step - loss: 10255.7471 - root_mean_squared_error: 101.2707
Epoch 8/200
191/191 ━━━━━━━━━━━━━━━━━━━━ 0s 627us/step - loss: 10367.5371 - root_mean_squared_error: 101.8211
Epoch 9/200
191/191 ━━━━━━━━━━━━━━━━━━━━ 0s 634us/step - loss: 10161.0684 - root_mean_squared_error: 100.8021
Epoch 10/2